In [1223]:
import torch
import torch.nn as nn
import numpy as np

def im2col_multi(X, kernel_shape, stride=1, padding=(0, 0)):
    B = X.shape[0]
    kH, kW = kernel_shape

    if isinstance(padding, tuple):
        pad_H, pad_W = padding
    else:
        pad_H = pad_W = padding

    X_padded = np.pad(X, ( (0, 0), (0, 0),(pad_H, pad_H), (pad_W, pad_W) ), mode='constant')

    H_p, W_p = X_padded.shape[2:]

    out_H = (H_p- kH) // stride + 1
    out_W = (W_p- kW) // stride + 1

    cols = []

    for b in range(B):
        for i in range(0, out_H*stride, stride):
            for j in range(0, out_W * stride, stride):
                patch = X_padded[b, :, i:i+kH, j:j+kW].ravel()
                cols.append(patch)
    
    return np.array(cols), out_H, out_W

def col2im_multi(cols, output_shape, kernel_shape, stride=1, padding=0):
    B, C, H, W = output_shape
    kH, kW = kernel_shape
    H_p, W_p = H+2*padding, W+2*padding
    X_padded = np.zeros((B, C, H_p, W_p))

    out_H = (H_p - kH)//stride + 1
    out_W = (W_p - kW)//stride + 1

    idx = 0
    for b in range(B):
        for i in range(0, out_H*stride, stride):
            for j in range(0, out_W*stride, stride):
                patch = cols[idx].reshape(C, kH, kW)
                X_padded[b, :, i:i+kH, j:j+kW] += patch
                idx += 1

    if padding>0:
        X_padded = X_padded[:, :, padding:-padding, padding:-padding]

    return X_padded

def conv2d_im2col_multi(X, W, stride=1, padding=0):
    
    C_out, C_in, kH, kW = W.shape
    B = X.shape[0]
    X_col, out_H, out_W = im2col_multi(X, (kH, kW), stride, padding)

    W_col = W.reshape(C_out, -1)

    print(f'x: {X_col.shape}, W: {W_col.shape}')

    Y_col = X_col @ W_col.T

    Y = Y_col.T.reshape(B, C_out, out_H, out_W)
    return Y

def conv_transpose2d_img2col_multi(Y, W, stride=1, padding=0, output_shape=None):
    C_out, C_in, kH, kW = W.shape
    B = Y.shape[0]
    Y_col = Y.reshape(C_out, -1)
    W_col = W.reshape(C_out, -1)
    X_col = W_col.T @ Y_col

    if output_shape is None:
        H_out = (Y.shape[2]-1) * stride - 2*padding + kH
        W_out = (Y.shape[3]-1) * stride - 2*padding + kW
        output_shape = (B, C_in, H_out, W_out)

    X = col2im_multi(X_col.T, output_shape=output_shape, kernel_shape=(kH, kW), stride=stride, padding=padding)

    return X

In [ ]:
#torch.manual_seed(1)

B= 10
C_out, C_in = 9, 5
input_size=14
kernel_size=6

stride=2
padding=3

x = torch.randn(B, C_in, input_size, input_size)

print(f'x: {x.shape}')

x: torch.Size([10, 5, 14, 14])


In [1225]:
conv = nn.Conv2d(C_in, C_out, kernel_size=kernel_size, stride=stride, padding=padding, bias=False, dilation=1)
convt = nn.ConvTranspose2d(C_out, C_in, kernel_size=kernel_size, stride=stride, padding=padding, bias=False)

#with torch.no_grad():
    #conv.weight.copy_(kernel)
    #conv.bias.zero_()
    #convt.weight.copy_(kernel)
    #convt.bias.zero_()

output_conv = conv(x)
print(f'Conv out: {output_conv.shape}')

output_convt = convt(output_conv)
print(f'ConvT out: {output_convt.shape}')

Conv out: torch.Size([10, 9, 8, 8])
ConvT out: torch.Size([10, 5, 14, 14])


In [1226]:
x = np.array(x)
kernel1 = conv.weight.detach().numpy()
kernel2 = convt.weight.detach().numpy()

print(f'kernel shapes: {kernel1.shape}, {kernel2.shape}')

kernel shapes: (9, 5, 6, 6), (9, 5, 6, 6)


C:\Users\Korisnik\AppData\Local\Temp\ipykernel_18128\1592921002.py:1: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  x = np.array(x)


In [1227]:
result = conv2d_im2col_multi(x, kernel1, stride=stride, padding=padding)
print(f'result: {result.shape}')

x: (640, 180), W: (9, 180)
result: (10, 9, 8, 8)


In [1228]:
original = conv_transpose2d_img2col_multi(result, kernel2, stride=stride, padding=padding, output_shape=None)
print(f'original: {original.shape}')

original: (10, 5, 14, 14)


In [1229]:
print(output_conv[-3, -3, -3])
print(result[-3, -3, -3])
print(np.allclose(output_conv.detach().numpy(), result, atol=1e-1, rtol=1e-1))
print(np.max(np.abs(output_conv.detach().numpy() - result)))

tensor([-0.1125,  0.3829,  0.0499,  1.3267,  0.0071, -0.2589,  0.2851, -0.0582],
       grad_fn=<SelectBackward0>)
[-0.6728334   0.23742473 -0.42635897 -0.4039906  -0.51800114  0.59526116
  0.07090192 -0.28215355]
False
2.8216147


In [1230]:
print(output_convt[-1, -2,-3 ])
print(original[-1, -2, -3])
print(np.allclose(output_convt.detach().numpy(), original, atol=1e-5, rtol=1e-5))
print(np.max(np.abs(output_convt.detach().numpy() - original)))

tensor([ 0.0223,  0.2385, -0.0065, -0.1372, -0.2606, -0.2082,  0.3780,  0.2684,
         0.0925, -0.1359, -0.0567, -0.0927,  0.3613, -0.1167],
       grad_fn=<SelectBackward0>)
[ 0.02231786  0.23853397 -0.00651578 -0.13716992 -0.26059666 -0.20819638
  0.37796225  0.26835277  0.09245802 -0.13590916 -0.05672663 -0.09269215
  0.36127907 -0.11668917]
True
3.8137659430503845e-07
